In [1]:
import os
import re
import numpy as np
import xml.etree.ElementTree as ET
from collections import defaultdict

# --- Configuration ---
xml_dir = 'Images' # Change this to the absolute path if needed
sequence_length = 10
stride = 1
anomaly_classes = {'Handgun', 'Short_rifle', 'Knife'}
output_file = 'y_true_binary.npy'

print("--- Step 1: XML Discovery and Target Extraction ---\n")

def get_sort_keys(filename):
    """
    Regex to extract the Camera ID and Frame Number for strict chronological sorting.
    Example: Cam1-From09-23-00To10-03-25_Segment_0_x264_frame_1.xml
    """
    match = re.search(r'Cam(\d+).*?_frame_(\d+)\.xml', filename)
    if match:
        cam_id = int(match.group(1))
        frame_num = int(match.group(2))
        return (cam_id, frame_num)
    return (float('inf'), float('inf'))

# 1. Get all XML files and sort them
xml_files = [f for f in os.listdir(xml_dir) if f.endswith('.xml')]
xml_files.sort(key=get_sort_keys)

# Organize structurally by Camera ID to prevent cross-camera sequence contamination
camera_streams = defaultdict(list)
for f in xml_files:
    cam_id, _ = get_sort_keys(f)
    camera_streams[cam_id].append(os.path.join(xml_dir, f))

print(f"Discovered {len(xml_files)} total XML validation frames across {len(camera_streams)} cameras.\n")

final_sequence_labels = []

# 2 & 3. Label Extraction and Sequence Alignment
for cam_id, stream_files in camera_streams.items():
    print(f"Processing Camera {cam_id} stream ({len(stream_files)} raw frames)...")
    
    # Extract raw binary labels for the current stream
    stream_labels = []
    for filepath in stream_files:
        try:
            tree = ET.parse(filepath)
            root = tree.getroot()
            
            is_anomaly = 0
            # Scan the XML structure for object names matching weapon classes
            for obj in root.findall('object'):
                name_element = obj.find('name')
                if name_element is not None and name_element.text in anomaly_classes:
                    is_anomaly = 1
                    break # Optimize: if one weapon is found, no need to check other objects
                    
            stream_labels.append(is_anomaly)
        except Exception as e:
            print(f"Error parsing {filepath}: {e}")
            stream_labels.append(0)
            
    raw_len = len(stream_labels)
    
    # 3. Apply Sliding Window over the raw timeline labels
    if raw_len >= sequence_length:
        stream_sequences = []
        # Steps by 1 and restricts the end bound (drops the last 9 frames implicitly)
        for i in range(0, raw_len - sequence_length + 1, stride):
            window = stream_labels[i : i + sequence_length]
            
            # Anomaly Rule: If any frame inside the window is a 1, flag the sequence as 1
            seq_label = 1 if any(val == 1 for val in window) else 0
            stream_sequences.append(seq_label)
            
        final_sequence_labels.extend(stream_sequences)
        print(f"  -> Generated {len(stream_sequences)} aligned sequences.")
    else:
        print(f"  -> Stream too short for a complete sequence. Skipped.")

# 4. Final Matrix Export Output
y_true_binary_full = np.array(final_sequence_labels, dtype=int)

print("\n--- Summary & Export ---")
print(f"Final aligned ground truth array shape: {y_true_binary_full.shape}")
print(f"Distribution: Normal Sequences = {np.sum(y_true_binary_full == 0)}, Anomaly Sequences = {np.sum(y_true_binary_full == 1)}")

np.save(output_file, y_true_binary_full)
print(f"\nSuccess! Array statically finalized and saved as '{output_file}'.")

--- Step 1: XML Discovery and Target Extraction ---

Discovered 5149 total XML validation frames across 3 cameras.

Processing Camera 1 stream (607 raw frames)...
  -> Generated 598 aligned sequences.
Processing Camera 5 stream (1031 raw frames)...
  -> Generated 1022 aligned sequences.
Processing Camera 7 stream (3511 raw frames)...
  -> Generated 3502 aligned sequences.

--- Summary & Export ---
Final aligned ground truth array shape: (5122,)
Distribution: Normal Sequences = 2496, Anomaly Sequences = 2626

Success! Array statically finalized and saved as 'y_true_binary.npy'.
